# DuckGPT — 05. From `core/` to `lib/`

Side-by-side diff of the didactic and production variants, plus a wall-clock
benchmark for the most expensive ones.

| Concern | `core/` | `lib/` | Why `lib/` differs |
|---|---|---|---|
| BPE training | Python loops + `Counter` | numpy-vectorised `np.unique` over packed pairs | ~50× faster on long corpora |
| Attention | Manual `softmax(Q K^T / √d)` | `F.scaled_dot_product_attention` (FlashAttention when available) | avoids materialising the `(B, H, T, T)` tensor |
| Optimiser | `Adam` (clear, slow) | `AdamW` with in-place ops + fused step when on CUDA | fewer temporary tensors, less Python overhead |
| LR schedule | inline function call per step | same function, but driving a wrapper that exposes `.lr` setter | identical behaviour, cleaner integration |
| Training | single process, no AMP | DDP, mixed precision, gradient clipping, checkpoint resume | makes a full GPT-2 124M run feasible |
| Sampling | greedy + temperature + top-k | adds top-p, repetition penalty, streaming | better generations, real-time UX |

Below we benchmark the two main hotspots: BPE training and attention forward.

In [ ]:
import sys, time, math
from pathlib import Path
import numpy as np
import torch

sys.path.append('../core')
sys.path.append('../lib')
from importlib import import_module
core_tok = import_module('02_tokenizer')   # python-only BPE
lib_tok  = import_module('tokenizer')      # vectorised BPE
print('core merges fn:', core_tok.merge.__name__)
print('lib  pair-count fn (numpy):', lib_tok._np is not None)

In [ ]:
# BPE training benchmark on a 250 KB slice of Shakespeare.
data = Path('../data/shakespeare/input.txt')
text = (data.read_text(encoding='utf-8')[:250_000]
        if data.exists() else 'olá DuckGPT! '*20_000)
print(f'corpus: {len(text):,} chars')

# core/ tokenizer with vocab 384 (12 merges over 256 bytes feels tiny; bump it)
from collections import Counter
ids = list(text.encode('utf-8'))
vocab = {i: bytes([i]) for i in range(256)}
merges = []
t0 = time.perf_counter()
for step in range(128):
    pair, _ = Counter(zip(ids, ids[1:])).most_common(1)[0]
    new_id = 256 + step
    ids = core_tok.merge(ids, pair, new_id)
    merges.append(pair); vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
core_time = time.perf_counter() - t0
print(f'core/ BPE 128 merges: {core_time:.2f}s')

In [ ]:
# lib/ tokenizer with the same corpus + vocab_size 384
tok = lib_tok.BPETokenizer(special_tokens={'<|endoftext|>': -1})
t0 = time.perf_counter()
tok.train(text, vocab_size=384)
lib_time = time.perf_counter() - t0
print(f'lib/  BPE 128 merges: {lib_time:.2f}s   (speed-up: {core_time/lib_time:.1f}x)')

In [ ]:
# Attention benchmark: manual softmax vs F.scaled_dot_product_attention
import torch, torch.nn.functional as F
torch.manual_seed(0)
B, H, T, D = 4, 8, 256, 64
q = torch.randn(B, H, T, D)
k = torch.randn(B, H, T, D)
v = torch.randn(B, H, T, D)
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))

def manual_attn(q, k, v, mask):
    s = (q @ k.transpose(-2, -1)) / math.sqrt(D)
    s = s.masked_fill(~mask, float('-inf'))
    return F.softmax(s, dim=-1) @ v

for fn, label in [(lambda: manual_attn(q, k, v, mask), 'manual softmax'),
                  (lambda: F.scaled_dot_product_attention(q, k, v, is_causal=True), 'F.sdpa')]:
    # warmup
    fn(); fn()
    t0 = time.perf_counter()
    for _ in range(20): fn()
    print(f'{label:>16}: {(time.perf_counter() - t0)/20*1e3:.2f} ms / call')

## Takeaway

Most of the speed-up from `core/` to `lib/` comes from:

1. Vectorising tight loops (BPE pair counting).
2. Letting PyTorch call into FlashAttention via `F.scaled_dot_product_attention`.
3. Mixed precision + larger batches on a GPU (not benchmarked here).

The model definition itself is intentionally the same shape — same blocks,
same dimensions, same maths. Once you can read `core/05_model.py`, the
`lib/model.py` is a small refactor to read on top.